# Retrieval-guided master-prompt selection

## 1. Goal and single-split contract

This notebook is a compact, reproducible experiment walkthrough. One run
evaluates exactly one configured split:

- `validation` maps to the Bias-in-Bios `dev` source;
- `test` maps to the Bias-in-Bios `test` source;
- the unselected split is never embedded, predicted, scored, ranked, or plotted.

Every configured condition is evaluated on the selected split and ranked within
language model. Complete CSVs, plots, prompts, and factor contrasts are saved in
the run directory. The notebook displays only compact preflight information,
exact prompt previews, current-split winners, and one selected factor-contrast
summary.

## 2. Environment and imports

Open this notebook from the repository root with any Python kernel that has
the packages in `requirements.txt` installed. IPython autoreload applies edits
to local Python modules before later cells run.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import JSON, Markdown, display

from configuration import load_config, validate_config
from dataset import (
    load_source_rows,
    select_profession_splits,
    select_run_data,
    task_settings,
)
from evaluation import resolve_metric_column
from modeling import build_prompt
from pipeline import (
    calculate_metrics,
    load_inference_run,
    prepare_embedding_cache,
    run_inference,
)

In [3]:
PROJECT_ROOT = Path.cwd().resolve()
CONFIG_PATH = PROJECT_ROOT / 'config.yaml'

if not CONFIG_PATH.exists():
    raise FileNotFoundError('Open this notebook from the repository root.')

## 3. Load and validate the configuration

`config.yaml` selects prompt files, while each file owns its complete prompt wording.
Configuration loading resolves those files before validation and inference.
The checked-in run uses deterministic `generated_output`; the separate
log-probability diagnostic remains supported but is not an experimental factor
and is never mixed into these within-run contrasts.

In [4]:
config = load_config(CONFIG_PATH)
validate_config(config)

evaluation_split = config['defaults']['evaluation_split']
prediction_method = config['inference']['prediction_method']
target, audit_column, professions, target_labels = task_settings(config)

display(Markdown(
    f'**Configured run:** hold out `{target}` and evaluate `{evaluation_split}`; '
    f'the language model receives `hard_text + {audit_column}` and uses '
    f'`prediction_method: {prediction_method}`.'
))

**Configured run:** hold out `profession` and evaluate `validation`; the language model receives `hard_text + gender` and uses `prediction_method: generated_output`.

## 4. Condition count and runtime controls

Zero-shot contributes one retrieval-independent condition per language model
and prompt. Positive example counts cross retrieval methods, embedding models,
example orders, prompts, and language models. The evaluation-row multiplier is
resolved after source loading; the flow always shows both the maximum balanced
capacity and the selected value.

In [5]:
retrieval = config['retrieval']
prompt_count = len(config['prompt_templates'])
positive_example_count_count = sum(example_count > 0 for example_count in retrieval['example_counts'])
zero_shot_conditions_per_model = (prompt_count if 0 in retrieval['example_counts'] else 0)
few_shot_conditions_per_model = (
    len(retrieval['methods'])
    * len(retrieval['embedding_models'])
    * positive_example_count_count
    * len(retrieval['example_orders'])
    * prompt_count
)
conditions_per_model = zero_shot_conditions_per_model + few_shot_conditions_per_model
language_model_count = len(config['inference']['language_models'])

pd.DataFrame([{
    'language_models': language_model_count,
    'conditions_per_language_model': conditions_per_model,
    'total_conditions': language_model_count * conditions_per_model,
    'configured_evaluation_per_profession_gender': (
        config['dataset']['evaluation_per_profession_gender']
    ),
}])

,language_models,conditions_per_language_model,total_conditions,configured_evaluation_per_profession_gender
0,3,50,150,5


## 5. Load sources and resolve evaluation capacity

`select_run_data()` applies the retrieval-pool cap and balanced evaluation
selection. The notebook shows the maximum available balanced cell size even
when an explicit smaller value is configured.

In [6]:
source_rows = load_source_rows(config, PROJECT_ROOT)
source_splits = select_profession_splits(config, source_rows)
(
    train_rows,
    evaluation_rows,
    selected_evaluation_per_cell,
    max_balanced_evaluation_per_cell,
) = select_run_data(config, source_splits)

display(Markdown(
    f'**Maximum balanced `{evaluation_split}` capacity:** '
    f'`{max_balanced_evaluation_per_cell}` rows per profession-gender cell  \n'
    f'**Selected `{evaluation_split}` size:** '
    f'`{selected_evaluation_per_cell}` rows per profession-gender cell'
))

assert {row['split'] for row in evaluation_rows} == {evaluation_split}
assert selected_evaluation_per_cell <= max_balanced_evaluation_per_cell
assert len(evaluation_rows) == len(professions) * 2 * selected_evaluation_per_cell

Selecting evaluation cells:   0%|          | 0/8 [00:00<?, ?cell/s]

**Maximum balanced `validation` capacity:** `988` rows per profession-gender cell  
**Selected `validation` size:** `5` rows per profession-gender cell

## 6. Preview the language-model input

The preview shows the exact zero-shot format when configured and the maximum
positive-count format. Actual retrieved prompts are saved in the complete
prediction CSV.

In [7]:
preview_template_name, preview_template = next(iter(config['prompt_templates'].items()))
configured_example_counts = config['retrieval']['example_counts']
positive_preview_counts = [
    example_count for example_count in configured_example_counts if example_count > 0
]
preview_example_counts = [0] if 0 in configured_example_counts else []
if positive_preview_counts:
    preview_example_counts.append(max(positive_preview_counts))

for preview_example_count in preview_example_counts:
    preview_messages = build_prompt(
        evaluation_rows[0],
        train_rows[:preview_example_count],
        target,
        target_labels,
        preview_template,
    )
    display(Markdown(
        f'**Template:** `{preview_template_name}`; **Examples:** `{preview_example_count}`'
    ))
    display(JSON(preview_messages, expanded=True))

**Template:** `neutral`; **Examples:** `0`

<IPython.core.display.JSON object>

**Template:** `neutral`; **Examples:** `8`

<IPython.core.display.JSON object>

## 7. Prepare the complete training-embedding tables

Positive-count runs require complete revision-pinned manifested training-embedding
tables. Missing, incompatible, or unmanifested tables are rebuilt here. Fingerprinted
evaluation-query-vector NPZ files are prepared or reused during inference; retrieval
selections are recomputed in memory for each required run. All-zero runs skip this step.

In [8]:
has_positive_example_count = any(example_count > 0 for example_count in config['retrieval']['example_counts'])
prepared_embedding_rows = (
    prepare_embedding_cache(config, PROJECT_ROOT, progress=print)
    if has_positive_example_count
    else {}
)

prepared_embedding_rows

Using device: mps
Preparing 257478 canonical training rows


Preparing complete embedding tables:   0%|          | 0/2 [00:00<?, ?model/s]

Reusing 257478 manifested training embeddings from /Users/AmirMohammad/Documents/Prompt Selection (B.Sc. Project)/retrieval-guided-master-prompt-selection/data/lancedb/semantic_qwen_qwen3-embedding-8b
Reusing 257478 manifested training embeddings from /Users/AmirMohammad/Documents/Prompt Selection (B.Sc. Project)/retrieval-guided-master-prompt-selection/data/lancedb/semantic_baai_bge-large-en-v1.5
Prepared complete manifested embedding tables at /Users/AmirMohammad/Documents/Prompt Selection (B.Sc. Project)/retrieval-guided-master-prompt-selection/data/lancedb


{'Qwen/Qwen3-Embedding-8B': 257478, 'BAAI/bge-large-en-v1.5': 257478}

## 8. Run inference

This is the expensive section. Before loading any missing language model, it
prepares or reuses cached query vectors, then recomputes the configured retrieval
methods in memory by exhaustively scoring every eligible training vector and releases
the retrieval matrices. It
then loads each missing language model and generates raw predictions. Fully resumed
and all-zero runs skip retrieval. This section does **not** calculate metrics,
rankings, contrasts, or plots. Before model inference starts, the cell prints both
the maximum balanced capacity and the selected cell size.

The combined predictions and the two count tables are saved under
`incomplete_run`. The final timestamped run contains the same three CSVs. Keep
the prediction CSV path if you want to recalculate metrics after restarting the
kernel.

In [ ]:
inference_run = run_inference(config, PROJECT_ROOT, progress=print)

Incomplete-run checkpoints: /Users/AmirMohammad/Documents/Prompt Selection (B.Sc. Project)/retrieval-guided-master-prompt-selection/results/incomplete_run
Holding out profession; language model input is hard_text + gender
Prediction method: generated_output


Selecting evaluation cells:   0%|          | 0/8 [00:00<?, ?cell/s]

Maximum balanced validation capacity before inference: 988 rows per profession/gender cell
Selected validation size before inference: 5 rows per profession/gender cell
Loaded 211585 filtered source rows; selected 137525 train and 40 validation rows for this run


Evaluating language models on validation:   0%|          | 0/3 [00:00<?, ?model/s]

Using device: mps


...

Predicting validation rows:   0%|          | 0/40 [00:00<?, ?row/s]

Predicting validation rows:   0%|          | 0/40 [00:00<?, ?row/s]

Predicting validation rows:   0%|          | 0/40 [00:00<?, ?row/s]

Predicting validation rows:   0%|          | 0/40 [00:00<?, ?row/s]

Predicting validation rows:   0%|          | 0/40 [00:00<?, ?row/s]

## 9. Calculate metrics and save artifacts

This section never loads a language model, embeds a query, retrieves examples,
or generates predictions. In the same kernel it uses `inference_run` directly.
After a kernel restart, run the setup and configuration cells, set
`saved_predictions_path` to the previous run's `<split>_predictions.csv`, and
jump here. The two count CSVs are loaded from the same directory.

The saved predictions must come from the same inference configuration. You may
change `ranking_metric`, `ranking_direction`, or metric code. Each recalculation
writes a new timestamped artifact directory.

In [ ]:
# After a completed run, replace the below path with that run's predictions CSV.
# saved_predictions_path = (
#     PROJECT_ROOT
#     / config['defaults']['output_dir']
#     / 'incomplete_run'
#     / f'{evaluation_split}_predictions.csv'
# )
#
# inference_run = load_inference_run(
#     config,
#     saved_predictions_path,
#     project_root=PROJECT_ROOT,
#     progress=print,
# )

In [ ]:
run = calculate_metrics(config, inference_run, PROJECT_ROOT, progress=print)

In [ ]:
assert run['evaluation_split'] == evaluation_split
assert set(run['predictions']['evaluation_split']) == {evaluation_split}
assert {'factor_contrast_details', 'factor_contrast_summary'} <= set(run)

display(Markdown(f'**Artifacts:** `{run["run_dir"]}`'))

## 10. Compact winners and factor contrasts

Only one winner per language model and one selected contrast slice are shown.
Edit `comparison_factor` or `comparison_metric` to inspect another saved
summary without recomputing predictions, metrics, or plots.

In [ ]:
metric_column = resolve_metric_column(config['defaults']['ranking_metric'])
ranking_columns = [
    'evaluation_split',
    'language_model',
    'rank',
    'is_best',
    'condition',
    metric_column,
]
best_conditions = run['results'].loc[run['results']['is_best'], ranking_columns].copy()
assert len(best_conditions) == len(config['inference']['language_models'])
best_conditions

In [ ]:
comparison_factor = 'prompt_name'
comparison_metric = metric_column

contrast_summary = run['factor_contrast_summary']
selected_contrasts = contrast_summary.loc[
    contrast_summary['factor'].eq(comparison_factor)
    & contrast_summary['metric'].eq(comparison_metric)
].copy()
if selected_contrasts.empty:
    raise ValueError(
        f'No saved contrasts exist for factor={comparison_factor!r} and '
        f'metric={comparison_metric!r}'
    )

contrast_display_columns = [
    'contrast_type',
    'aggregation_scope',
    'scope_language_model',
    'from_factor_value',
    'to_factor_value',
    'direction',
    'n_defined_pairs',
    'n_total_pairs',
    'mean_from_metric_value',
    'mean_to_metric_value',
    'mean_delta',
    'std_delta',
    'n_improved',
    'n_tied',
    'n_worsened',
    'improvement_rate',
]
selected_contrasts[contrast_display_columns]

## 11. Metric reference and next-run guidance

### 1. Notation, counts, and supports

For one prediction condition, $N$ is `sample_count`, $K$ is `n_target_labels`,
and $G$ is `n_audit_groups`. Row $i$ has true target label $y_i$, predicted
target label $\hat y_i$, and audit group $a_i$. Target label $c$ is evaluated
one-vs-rest: $c$ is positive and every other target label is negative. Audit
group $g$ contains $N_g$ rows, stored as `audit_group_n`.

$$
\begin{aligned}
TP_c&=\sum_i\mathbf{1}[y_i=c\land\hat y_i=c], &
FP_c&=\sum_i\mathbf{1}[y_i\ne c\land\hat y_i=c],\\
FN_c&=\sum_i\mathbf{1}[y_i=c\land\hat y_i\ne c], &
TN_c&=\sum_i\mathbf{1}[y_i\ne c\land\hat y_i\ne c].
\end{aligned}
$$

These four counts are stored as `tp`, `fp`, `fn`, and `tn`. The remaining
detailed output columns are:

- `positive_support`: $n_c=TP_c+FN_c$;
- `negative_support`: $FP_c+TN_c=N-n_c$;
- `predicted_positive`: $TP_c+FP_c$;
- `selection_rate`: $SR_c=(TP_c+FP_c)/N$.

The audit-group table applies exactly the same definitions after restricting
the sums to $a_i=g$; its counts use the subscript $(c,g)$ and its selection-rate
denominator is $N_g$. The long-form confusion-matrix output stores `count` as

$$
C_{r,s}=\sum_i\mathbf{1}[y_i=r\land\hat y_i=s]
$$

for every configured true-label row $r$ and predicted-label column $s$.

### 2. Target-label and audit-group rates

For each target label, and identically within each audit group:

- `precision`: $PPV_c=TP_c/(TP_c+FP_c)$;
- `recall`: $TPR_c=TP_c/(TP_c+FN_c)$;
- `f1`: $F1_c=2TP_c/(2TP_c+FP_c+FN_c)$;
- `specificity`: $TNR_c=TN_c/(TN_c+FP_c)$;
- `false_positive_rate`: $FPR_c=FP_c/(FP_c+TN_c)$;
- `false_negative_rate`: $FNR_c=FN_c/(FN_c+TP_c)$;
- `negative_predictive_value`: $NPV_c=TN_c/(TN_c+FN_c)$.

`audit_group_accuracy` is the multiclass accuracy repeated for each target-label
row of the same audit group:

$$
Accuracy_g=\frac{\sum_{i:a_i=g}\mathbf{1}[y_i=\hat y_i]}{N_g}.
$$

Whenever their denominators are nonzero, $FPR=1-Specificity$ and
$FNR=1-Recall$. Higher selection rate is not inherently better or worse;
precision, recall, F1, specificity, NPV, and audit-group accuracy are better
when higher, while FPR and FNR are better when lower.

The implementation returns `NaN` for a rate whose required denominator is
zero. Macro and weighted aggregates omit undefined rates; a coverage output
reports how many values remained. Selection rate and audit-group accuracy are
defined because validated conditions and observed audit groups are nonempty.

### 3. Overall quality and agreement

For any target-label rate $m_c$, let $D_m$ contain the target labels where it is
defined, and let $D_m^+$ additionally require positive true support $n_c>0$:

$$
\begin{aligned}
Accuracy&=\frac{\sum_cTP_c}{N},\\
Macro(m)&=\frac{1}{|D_m|}\sum_{c\in D_m}m_c,\\
Weighted(m)&=\frac{\sum_{c\in D_m^+}n_cm_c}
{\sum_{c\in D_m^+}n_c}.
\end{aligned}
$$

These formulas map to `macro_precision`, `macro_recall / balanced_accuracy`,
`macro_f1`, `weighted_precision`, and `weighted_f1`. The defined-label coverage
outputs are
`n_precision_defined_target_labels` $=|D_{Precision}|$,
`n_recall_defined_target_labels` $=|D_{Recall}|$, and
`n_f1_defined_target_labels` $=|D_{F1}|$. An aggregate is `NaN` when its defined
set is empty; a weighted aggregate also needs at least one positive weight.

In this single-label multiclass task, let $T=\sum_cTP_c$ be the number of
correct rows and $E=\sum_cFP_c=\sum_cFN_c$ the number of errors. Since
$N=T+E$:

$$
MicroPrecision=MicroRecall=MicroF1=Accuracy=\frac{T}{N},
$$

$$
WeightedRecall=\frac{1}{N}\sum_{c:n_c>0}n_c\frac{TP_c}{n_c}
=Accuracy,\qquad
BalancedAccuracy=\frac{1}{|D_{Recall}|}\sum_{c\in D_{Recall}}Recall_c
=MacroRecall.
$$

The results store those equality families once under
`accuracy / micro_precision / micro_recall / micro_f1 / weighted_recall` and
`macro_recall / balanced_accuracy`. Each individual standard name is still a
valid `ranking_metric` alias.

For multiclass agreement, let $C$ be the confusion matrix,
$s=\sum_{r,k}C_{r,k}=N$, $q=\operatorname{trace}(C)$,
$p_k=\sum_rC_{r,k}$ be predicted totals, and $t_k=\sum_rC_{k,r}$ be true totals:

The `matthews_correlation_coefficient` output is

$$
MCC=\frac{qs-\sum_kp_kt_k}
{\sqrt{(s^2-\sum_kp_k^2)(s^2-\sum_kt_k^2)}}.
$$

With observed agreement $p_o=Accuracy$ and chance-expected agreement
$p_e=\sum_k(t_k/s)(p_k/s)$:

The `cohen_kappa` output is

$$
\kappa=\frac{p_o-p_e}{1-p_e}.
$$

Higher accuracy, MCC, and kappa are better; one means perfect agreement. The
implementation uses scikit-learn's degenerate-case conventions: MCC is `0.0`
when its denominator is zero, while kappa is `NaN` when $1-p_e=0$.

### 4. Target-label fairness across audit groups

For target label $c$ and rate $m$, let $G_{m,c}$ contain the audit groups where
$m_{c,g}$ is defined. A range ignores undefined values and exists only when at
least two values remain:

$$
Range_g(m_{c,g})=\max_{g\in G_{m,c}}m_{c,g}
-\min_{g\in G_{m,c}}m_{c,g}.
$$

The `fairness_metrics` columns are:

- `demographic_parity_difference`: $DPDiff_c=Range_g(SR_{c,g})$;
- `demographic_parity_ratio`:
  $DPRatio_c=\min_{g\in G_{SR,c}}SR_{c,g}/\max_{g\in G_{SR,c}}SR_{c,g}$;
- `equal_opportunity_difference`: $EODiff_c=Range_g(TPR_{c,g})$;
- `false_positive_rate_difference`: $FPRDiff_c=Range_g(FPR_{c,g})$;
- `equalized_odds_difference`: $EOddsDiff_c=\max(EODiff_c,FPRDiff_c)$;
- `predictive_parity_difference`: $PPDiff_c=Range_g(PPV_{c,g})$.

A difference of zero and a demographic-parity ratio of one mean equality across
the compared audit groups. The ratio requires at least two defined selection
rates and a positive maximum; if every group has zero selection rate, it is
`NaN` rather than $0/0$. Equalized-odds difference is `NaN` unless both its TPR-
and FPR-range components are defined.

The per-target-label coverage outputs are:

- `n_audit_groups_compared` $=G$;
- `n_selection_rate_defined_audit_groups` $=|G_{SR,c}|$;
- `n_recall_defined_audit_groups` $=|G_{TPR,c}|$;
- `n_false_positive_rate_defined_audit_groups` $=|G_{FPR,c}|$;
- `n_precision_defined_audit_groups` $=|G_{PPV,c}|$.

### 5. Condition-level fairness summaries and coverage

Audit-group accuracy is summarized as:

- `worst_audit_group_accuracy`: $\min_gAccuracy_g$;
- `audit_group_accuracy_difference`: $Range_g(Accuracy_g)$.

The accuracy difference requires at least two audit groups. Higher worst-group
accuracy and a lower accuracy difference are preferable.

For each target-label fairness value $d_c$, let $D_d$ contain the target labels
where it is defined. Every fairness family receives the same three summaries:

$$
Mean(d)=\frac{1}{|D_d|}\sum_{c\in D_d}d_c,\qquad
Min(d)=\min_{c\in D_d}d_c,\qquad
Max(d)=\max_{c\in D_d}d_c.
$$

The exact result-column families are:

| Per-target-label value $d$       | Condition result columns                                                                                          |
|----------------------------------|-------------------------------------------------------------------------------------------------------------------|
| `demographic_parity_difference`  | `mean_demographic_parity_difference`, `min_demographic_parity_difference`, `max_demographic_parity_difference`    |
| `demographic_parity_ratio`       | `mean_demographic_parity_ratio`, `min_demographic_parity_ratio`, `max_demographic_parity_ratio`                   |
| `equal_opportunity_difference`   | `mean_equal_opportunity_difference`, `min_equal_opportunity_difference`, `max_equal_opportunity_difference`       |
| `false_positive_rate_difference` | `mean_false_positive_rate_difference`, `min_false_positive_rate_difference`, `max_false_positive_rate_difference` |
| `equalized_odds_difference`      | `mean_equalized_odds_difference`, `min_equalized_odds_difference`, `max_equalized_odds_difference`                |
| `predictive_parity_difference`   | `mean_predictive_parity_difference`, `min_predictive_parity_difference`, `max_predictive_parity_difference`       |

Each family omits undefined target-label values. If $D_d$ is empty, all three
summaries are `NaN`. Its exact coverage output is:

- `n_demographic_parity_defined_target_labels` $=|D_{DPDiff}|$;
- `n_demographic_parity_ratio_defined_target_labels` $=|D_{DPRatio}|$;
- `n_equal_opportunity_defined_target_labels` $=|D_{EODiff}|$;
- `n_false_positive_rate_defined_target_labels` $=|D_{FPRDiff}|$;
- `n_equalized_odds_defined_target_labels` $=|D_{EOddsDiff}|$;
- `n_predictive_parity_defined_target_labels` $=|D_{PPDiff}|$.

For difference metrics, zero is best, so their minimum is the best target-label
value and their maximum is the worst. For demographic-parity ratio, one is best,
so the minimum ratio is the worst target-label value and the maximum is the best.
Coverage plots show $K$ or $G$ first so every defined count has an explicit
denominator.

Quality and fairness must be interpreted together. With `target: profession`
and audit column `gender`, these are conventional protected-group fairness
diagnostics. With `target: gender` and audit column `profession`, the same
mathematics is better described as profession-conditioned performance or
stereotype/association diagnostics. Profession and gender remain separate
experiments with the same prompt candidates and separate winners.

### Factor-contrast interpretation

The saved factor-contrast details compare every pair of configured levels while
holding every other applicable factor fixed. The summary aggregates raw deltas
overall and, except for language-model contrasts, within language model. A
positive `improvement` means better performance after respecting the metric's
direction. Detail rows retain the factor transition, fixed JSON context,
source/target values, delta, outcome, and source/target condition counts. Summary
rows retain scope, total/defined pairs, mean source/target values, delta
mean/standard deviation, improved/tied/worsened counts, and improvement rate.
Undefined metric pairs remain in `n_total_pairs` but not `n_defined_pairs`.

Strict example-count contrasts use positive counts only. The separate
`zero_shot_to_few_shot` contrast averages the complete retrieval grid at each
positive count within one language-model and prompt context before comparing it
with zero-shot. These summaries are descriptive paired differences, not causal
estimates or significance tests.

### Next-run guidance

Keep `evaluation_split: validation` while developing prompt candidates. Change
it deliberately to `test` only when you intend to evaluate the complete
condition grid on test. Profession and gender are separate experiments with the
same prompt candidates and separate winners; do not mix their rankings.